# 🏅 Olympic Data Lake Pipeline

## RAW → BRONZE → PRATA (SILVER) → GOLD

Este notebook implementa um pipeline completo de Data Lake para datasets olímpicos,
seguindo boas práticas de engenharia de dados, reprodutibilidade e documentação.

### Camadas:
- **RAW**: Dados brutos originais (CSV + metadados JSON)
- **BRONZE**: Dados padronizados (Parquet + limpeza básica)
- **PRATA**: Dados integrados (JOINs + curadoria)
- **GOLD**: Dados analíticos (agregações + visualizações)

---
## 0. Setup e Importações

In [ ]:
import pandas as pd
import numpy as np
import json
import os
import re
import shutil
from pathlib import Path
from datetime import datetime
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use('Agg')
import seaborn as sns

sns.set_theme(style='whitegrid')

BASE_DIR = Path.cwd()
DATALAKE_DIR = BASE_DIR / 'olympics-datalake'
RAW_SOURCE_DIR = BASE_DIR / 'raw'
RAW_DIR = DATALAKE_DIR / 'raw'
BRONZE_DIR = DATALAKE_DIR / 'bronze'
PRATA_DIR = DATALAKE_DIR / 'prata'
GOLD_DIR = DATALAKE_DIR / 'gold'
PROCESSING_DATE = datetime.now().isoformat()

print(f'Base dir: {BASE_DIR}')
print(f'Data Lake dir: {DATALAKE_DIR}')
print(f'Processing date: {PROCESSING_DATE}')

### Funções Utilitárias

In [ ]:
def ensure_dir(path: Path) -> Path:
    path.mkdir(parents=True, exist_ok=True)
    return path


def normalize_column_name(col: str) -> str:
    col = col.strip().lower()
    col = re.sub(r'[^a-z0-9]+', '_', col)
    col = col.strip('_')
    return col


def save_metadata(metadata: dict, filepath: Path):
    with open(filepath, 'w', encoding='utf-8') as f:
        json.dump(metadata, f, ensure_ascii=False, indent=2, default=str)
    print(f'  Metadata salvo: {filepath.relative_to(DATALAKE_DIR)}')


def get_schema_info(df: pd.DataFrame) -> list:
    return [
        {'column': col, 'dtype': str(dtype), 'non_null_count': int(df[col].notna().sum())}
        for col, dtype in df.dtypes.items()
    ]


def load_csv_safe(filepath: Path) -> pd.DataFrame:
    for enc in ['utf-8', 'latin-1', 'cp1252']:
        try:
            df = pd.read_csv(filepath, encoding=enc, low_memory=False)
            print(f'  Carregado ({enc}): {df.shape[0]} linhas, {df.shape[1]} colunas')
            return df
        except (UnicodeDecodeError, UnicodeError):
            continue
    raise ValueError(f'Encoding error: {filepath}')


def validate_schema(df: pd.DataFrame, name: str):
    issues = []
    dup_cols = df.columns[df.columns.duplicated()].tolist()
    if dup_cols:
        issues.append(f'Colunas duplicadas: {dup_cols}')
    all_null = [col for col in df.columns if df[col].isna().all()]
    if all_null:
        issues.append(f'Colunas 100% nulas: {all_null}')
    dup_rows = df.duplicated().sum()
    if dup_rows > 0:
        issues.append(f'{dup_rows} linhas duplicadas')
    if issues:
        print(f'  AVISO {name}: {'; '.join(issues)}')
    else:
        print(f'  OK {name}: schema consistente')
    return issues


print('Funcoes utilitarias carregadas.')

### Criar Estrutura de Diretórios

In [ ]:
dirs = [
    RAW_DIR, BRONZE_DIR, PRATA_DIR,
    GOLD_DIR / 'analise_medalhas',
    GOLD_DIR / 'analise_modalidades',
    GOLD_DIR / 'analise_genero',
]
for d in dirs:
    ensure_dir(d)
    print(f'Criado: {d.relative_to(DATALAKE_DIR)}')

print('Estrutura de diretorios criada.')

---
## 1. Camada RAW — Ingestão de Dados Brutos

- Carregamos CSVs originais
- Validamos consistência de schema
- Copiamos para camada RAW do Data Lake
- Geramos metadados JSON

In [ ]:
RAW_DATASETS = {
    'olympics_historico': {
        'files': {
            'athlete_event_result': RAW_SOURCE_DIR / '1986-2022' / 'world_olympedia_olympics_athlete_event_result.csv',
            'game_medal_tally': RAW_SOURCE_DIR / '1986-2022' / 'world_olympedia_olympics_game_medal_tally(4).csv',
            'result': RAW_SOURCE_DIR / '1986-2022' / 'world_olympedia_olympics_result(2).csv',
        },
        'source': 'Olympedia / World Olympic Data (1896-2022)',
        'description': 'Dados historicos de Olimpiadas incluindo resultados de atletas, quadro de medalhas e detalhes de eventos.',
    },
    'olympics_paris2024': {
        'files': {
            'medallists': RAW_SOURCE_DIR / '2024' / 'medallists.csv',
            'medals': RAW_SOURCE_DIR / '2024' / 'medals(1).csv',
        },
        'source': 'Kaggle / Paris 2024 Olympics Dataset',
        'description': 'Dados das Olimpiadas de Paris 2024 incluindo medalhistas e quadro de medalhas.',
    },
}

print('Datasets mapeados:')
for group, info in RAW_DATASETS.items():
    print(f'  {group}: {list(info["files"].keys())}')

In [ ]:
raw_dataframes = {}

for group_name, group_info in RAW_DATASETS.items():
    print(f'\n{"="*60}')
    print(f'Processando grupo: {group_name}')
    print(f'{"="*60}')
    for ds_name, filepath in group_info['files'].items():
        key = f'{group_name}__{ds_name}'
        print(f'\n-> Dataset: {ds_name}')
        print(f'   Arquivo: {filepath.name}')
        df = load_csv_safe(filepath)
        validate_schema(df, ds_name)
        raw_dataframes[key] = df

print(f'\nTotal datasets carregados: {len(raw_dataframes)}')

In [ ]:
# Salvar CSVs na camada RAW
raw_csv_files = {
    'olympics_historico': raw_dataframes['olympics_historico__athlete_event_result'],
    'olympics_paris2024': raw_dataframes['olympics_paris2024__medallists'],
}

for name, df in raw_csv_files.items():
    csv_path = RAW_DIR / f'{name}.csv'
    df.to_csv(csv_path, index=False, encoding='utf-8')
    print(f'Salvo: {csv_path.relative_to(DATALAKE_DIR)}')

# Salvar como JSON
for name, df in raw_csv_files.items():
    json_path = RAW_DIR / f'{name}.json'
    df.to_json(json_path, orient='records', force_ascii=False, indent=2)
    print(f'Salvo: {json_path.relative_to(DATALAKE_DIR)}')

# Datasets auxiliares
aux_datasets = {
    'medal_tally_historico': raw_dataframes['olympics_historico__game_medal_tally'],
    'result_historico': raw_dataframes['olympics_historico__result'],
    'medals_paris2024': raw_dataframes['olympics_paris2024__medals'],
}
for name, df in aux_datasets.items():
    csv_path = RAW_DIR / f'{name}.csv'
    df.to_csv(csv_path, index=False, encoding='utf-8')
    print(f'Salvo (aux): {csv_path.relative_to(DATALAKE_DIR)}')

In [ ]:
# Metadados RAW
raw_metadata_defs = {
    'olympics_historico': {
        'dataset_name': 'olympics_historico',
        'source': 'Olympedia / World Olympic Data',
        'description': 'Resultados individuais de atletas em eventos olimpicos de 1896 a 2022.',
        'main_fields': ['edition', 'country_noc', 'sport', 'event', 'athlete', 'medal', 'position'],
        'collection_date': '2023-01-01',
        'observations': 'Campo medal possui ~86% de valores nulos (atletas sem medalha).',
    },
    'olympics_paris2024': {
        'dataset_name': 'olympics_paris2024',
        'source': 'Kaggle / Paris 2024 Olympics Dataset',
        'description': 'Medalhistas individuais das Olimpiadas de Paris 2024.',
        'main_fields': ['medal_type', 'name', 'gender', 'country_code', 'discipline', 'event'],
        'collection_date': '2024-08-15',
        'observations': 'Campos team/team_gender/code_team nulos para eventos individuais (~33%).',
    },
    'medal_tally_historico': {
        'dataset_name': 'medal_tally_historico',
        'source': 'Olympedia / World Olympic Data',
        'description': 'Quadro de medalhas por pais e edicao olimpica de 1896 a 2022.',
        'main_fields': ['year', 'edition', 'country', 'country_noc', 'gold', 'silver', 'bronze', 'total'],
        'collection_date': '2023-01-01',
        'observations': '1807 registros sem valores nulos.',
    },
    'result_historico': {
        'dataset_name': 'result_historico',
        'source': 'Olympedia / World Olympic Data',
        'description': 'Detalhes de resultados de eventos olimpicos.',
        'main_fields': ['result_id', 'event_title', 'edition', 'sport', 'result_date', 'result_location'],
        'collection_date': '2023-01-01',
        'observations': 'Campo result_location possui 1 valor nulo.',
    },
    'medals_paris2024': {
        'dataset_name': 'medals_paris2024',
        'source': 'Kaggle / Paris 2024 Olympics Dataset',
        'description': 'Quadro de medalhas de Paris 2024 com tipo, data, atleta, disciplina e pais.',
        'main_fields': ['medal_type', 'medal_date', 'name', 'gender', 'discipline', 'event', 'country_code'],
        'collection_date': '2024-08-15',
        'observations': '1044 registros. Campo url_event possui 9 nulos.',
    },
}

for ds_name, meta_base in raw_metadata_defs.items():
    df = raw_csv_files.get(ds_name, aux_datasets.get(ds_name))
    metadata = {
        **meta_base,
        'layer': 'raw',
        'schema': get_schema_info(df),
        'row_count': len(df),
        'column_count': len(df.columns),
        'processing_date': PROCESSING_DATE,
    }
    save_metadata(metadata, RAW_DIR / f'{ds_name}_metadata.json')

print('\nCamada RAW concluida.')

---
## 2. Camada BRONZE — Padronização de Dados

- CSV → Parquet
- Nomes de colunas em snake_case
- Limpeza básica (trimming, nulos)
- Metadados com transformações

In [ ]:
def process_bronze(df: pd.DataFrame, dataset_name: str) -> tuple:
    transformations = []

    # Normalizar nomes de colunas
    original_cols = list(df.columns)
    df.columns = [normalize_column_name(c) for c in df.columns]
    renamed = {o: n for o, n in zip(original_cols, df.columns) if o != n}
    if renamed:
        transformations.append(f'Colunas renomeadas: {renamed}')

    # Trimming de strings
    str_cols = df.select_dtypes(include='object').columns
    for col in str_cols:
        df[col] = df[col].astype(str).str.strip()
        df[col] = df[col].replace('nan', np.nan)
    if len(str_cols) > 0:
        transformations.append(f'Trimming em {len(str_cols)} colunas string')

    # Remover duplicatas
    n_dups = df.duplicated().sum()
    if n_dups > 0:
        df = df.drop_duplicates().reset_index(drop=True)
        transformations.append(f'{n_dups} duplicatas removidas')

    # Padronizar booleanos em strings
    for col in df.columns:
        if df[col].dtype == 'object':
            unique_vals = set(df[col].dropna().str.lower().unique())
            if unique_vals.issubset({'true', 'false'}):
                df[col] = df[col].str.lower().map({'true': True, 'false': False})
                transformations.append(f'Coluna {col} convertida para boolean')

    return df, transformations


print('Funcao process_bronze definida.')

In [ ]:
all_raw = {**raw_csv_files, **aux_datasets}
bronze_dataframes = {}

for name, df_raw in all_raw.items():
    print(f'\n{"="*50}')
    print(f'Bronze: {name}')
    print(f'{"="*50}')

    df = df_raw.copy()
    df_bronze, transforms = process_bronze(df, name)

    parquet_path = BRONZE_DIR / f'{name}.parquet'
    df_bronze.to_parquet(parquet_path, index=False, engine='pyarrow')
    print(f'  Parquet salvo: {parquet_path.relative_to(DATALAKE_DIR)}')
    print(f'  Shape: {df_bronze.shape}')

    metadata = {
        'dataset_name': name,
        'layer': 'bronze',
        'source_dataset': f'raw/{name}.csv',
        'schema': get_schema_info(df_bronze),
        'transformations_applied': transforms,
        'row_count': len(df_bronze),
        'column_count': len(df_bronze.columns),
        'processing_date': PROCESSING_DATE,
    }
    save_metadata(metadata, BRONZE_DIR / f'{name}_metadata.json')
    bronze_dataframes[name] = df_bronze

print(f'\nCamada BRONZE concluida. {len(bronze_dataframes)} datasets.')

In [ ]:
# Inspecionar schemas Bronze
for name, df in bronze_dataframes.items():
    print(f'\n{name}:')
    print(f'  Colunas: {list(df.columns)}')
    print(f'  Shape: {df.shape}')

---
## 3. Camada PRATA (SILVER) — Integração de Dados

Datasets curados:
1. **medalhas_1986_2024**: Quadro de medalhas unificado
2. **modalidades_1986_2024**: Esportes/eventos ao longo dos anos
3. **atletas_por_sexo**: Participação e medalhas por gênero

### 3.1 Dataset: medalhas_1986_2024

In [ ]:
df_tally_hist = bronze_dataframes['medal_tally_historico'].copy()
print(f'Medal tally historico: {df_tally_hist.shape}')
print(f'Colunas: {list(df_tally_hist.columns)}')
df_tally_hist.head(3)

In [ ]:
# Paris 2024: agregar medalhas por pais
df_medals_2024 = bronze_dataframes['medals_paris2024'].copy()

df_medals_2024['gold'] = (df_medals_2024['medal_type'].str.lower().str.contains('gold')).astype(int)
df_medals_2024['silver'] = (df_medals_2024['medal_type'].str.lower().str.contains('silver')).astype(int)
df_medals_2024['bronze_medal'] = (df_medals_2024['medal_type'].str.lower().str.contains('bronze')).astype(int)

tally_2024 = df_medals_2024.groupby(['country_code', 'country', 'country_long']).agg(
    gold=('gold', 'sum'),
    silver=('silver', 'sum'),
    bronze=('bronze_medal', 'sum'),
).reset_index()
tally_2024['total'] = tally_2024['gold'] + tally_2024['silver'] + tally_2024['bronze']
tally_2024['year'] = 2024
tally_2024['edition'] = '2024 Summer Olympics'
tally_2024['edition_id'] = 9999

tally_2024 = tally_2024.rename(columns={'country_code': 'country_noc', 'country_long': 'country'})
tally_2024 = tally_2024.drop(columns=['country'], errors='ignore')
tally_2024 = tally_2024.rename(columns={'country': 'country'})

print(f'Medal tally Paris 2024: {tally_2024.shape}')
tally_2024.sort_values('total', ascending=False).head(5)

In [ ]:
# Unificar medalhas
common_cols = ['year', 'edition', 'edition_id', 'country_noc', 'gold', 'silver', 'bronze', 'total']

hist_cols = [c for c in common_cols if c in df_tally_hist.columns]
tally_2024_cols = [c for c in common_cols if c in tally_2024.columns]

if 'country' in df_tally_hist.columns:
    hist_cols = ['country'] + [c for c in hist_cols if c != 'country']

df_medalhas = pd.concat([
    df_tally_hist[hist_cols],
    tally_2024[tally_2024_cols],
], ignore_index=True)

before = len(df_medalhas)
df_medalhas = df_medalhas.drop_duplicates().reset_index(drop=True)
after = len(df_medalhas)
print(f'medalhas_1986_2024: {after} linhas (removidas {before - after} duplicatas)')

prata_path = PRATA_DIR / 'medalhas_1986_2024.parquet'
df_medalhas.to_parquet(prata_path, index=False, engine='pyarrow')
print(f'Salvo: {prata_path.relative_to(DATALAKE_DIR)}')
df_medalhas.head()

### 3.2 Dataset: modalidades_1986_2024

In [ ]:
df_hist = bronze_dataframes['olympics_historico'].copy()

modalidades_hist = df_hist[['edition', 'edition_id', 'sport', 'event']].drop_duplicates()
modalidades_hist['year'] = modalidades_hist['edition'].str.extract(r'(\d{4})').astype(int)
modalidades_hist['source'] = 'historico'
print(f'Modalidades historicas: {len(modalidades_hist)}')

df_2024 = bronze_dataframes['olympics_paris2024'].copy()
modalidades_2024 = df_2024[['discipline', 'event']].drop_duplicates()
modalidades_2024 = modalidades_2024.rename(columns={'discipline': 'sport'})
modalidades_2024['year'] = 2024
modalidades_2024['edition'] = '2024 Summer Olympics'
modalidades_2024['edition_id'] = 9999
modalidades_2024['source'] = 'paris2024'
print(f'Modalidades Paris 2024: {len(modalidades_2024)}')

df_modalidades = pd.concat([
    modalidades_hist[['year', 'edition', 'edition_id', 'sport', 'event', 'source']],
    modalidades_2024[['year', 'edition', 'edition_id', 'sport', 'event', 'source']],
], ignore_index=True)
df_modalidades = df_modalidades.drop_duplicates().reset_index(drop=True)
print(f'\nmodalidades_1986_2024: {len(df_modalidades)} registros')

prata_path = PRATA_DIR / 'modalidades_1986_2024.parquet'
df_modalidades.to_parquet(prata_path, index=False, engine='pyarrow')
print(f'Salvo: {prata_path.relative_to(DATALAKE_DIR)}')
df_modalidades.head()

### 3.3 Dataset: atletas_por_sexo

In [ ]:
df_hist_sex = bronze_dataframes['olympics_historico'].copy()
df_hist_sex['year'] = df_hist_sex['edition'].str.extract(r'(\d{4})').astype(int)

def infer_gender(event_str):
    event_lower = str(event_str).lower()
    if 'women' in event_lower or 'ladies' in event_lower:
        return 'Female'
    elif 'men' in event_lower or "men's" in event_lower:
        return 'Male'
    else:
        return 'Mixed/Unknown'

df_hist_sex['gender'] = df_hist_sex['event'].apply(infer_gender)
df_hist_sex['has_medal'] = df_hist_sex['medal'].notna().astype(int)

hist_gender = df_hist_sex.groupby(['year', 'edition', 'gender']).agg(
    total_participations=('athlete', 'count'),
    unique_athletes=('athlete_id', 'nunique'),
    total_medals=('has_medal', 'sum'),
).reset_index()
hist_gender['source'] = 'historico'
print(f'Genero (historico): {len(hist_gender)} registros')

df_2024_sex = bronze_dataframes['olympics_paris2024'].copy()
paris_gender = df_2024_sex.groupby(['gender']).agg(
    total_participations=('name', 'count'),
    unique_athletes=('code_athlete', 'nunique'),
    total_medals=('is_medallist', 'sum'),
).reset_index()
paris_gender['year'] = 2024
paris_gender['edition'] = '2024 Summer Olympics'
paris_gender['source'] = 'paris2024'
print(f'Genero (Paris 2024): {len(paris_gender)} registros')

common = ['year', 'edition', 'gender', 'total_participations', 'unique_athletes', 'total_medals', 'source']
df_genero = pd.concat([hist_gender[common], paris_gender[common]], ignore_index=True)
df_genero = df_genero.drop_duplicates().reset_index(drop=True)
print(f'\natletas_por_sexo: {len(df_genero)} registros')

prata_path = PRATA_DIR / 'atletas_por_sexo.parquet'
df_genero.to_parquet(prata_path, index=False, engine='pyarrow')
print(f'Salvo: {prata_path.relative_to(DATALAKE_DIR)}')
df_genero.head(10)

In [ ]:
# Metadados PRATA
prata_metadata = {
    'medalhas_1986_2024': {
        'dataset_name': 'medalhas_1986_2024',
        'layer': 'prata',
        'source_datasets': ['bronze/medal_tally_historico', 'bronze/medals_paris2024'],
        'join_logic': 'UNION dos quadros de medalhas historico e Paris 2024 com colunas comuns.',
        'derived_fields': ['Nenhum campo derivado adicional'],
        'data_quality_notes': 'Deduplicacao aplicada. Dados de 2024 agregados de medalhas individuais.',
        'schema': get_schema_info(df_medalhas),
        'row_count': len(df_medalhas),
        'processing_date': PROCESSING_DATE,
    },
    'modalidades_1986_2024': {
        'dataset_name': 'modalidades_1986_2024',
        'layer': 'prata',
        'source_datasets': ['bronze/olympics_historico', 'bronze/olympics_paris2024'],
        'join_logic': 'UNION de sport/event unicos do historico com discipline/event de Paris 2024.',
        'derived_fields': ['year (extraido da edicao)', 'source (historico/paris2024)'],
        'data_quality_notes': 'Deduplicacao aplicada. Nomes de esportes podem diferir entre fontes.',
        'schema': get_schema_info(df_modalidades),
        'row_count': len(df_modalidades),
        'processing_date': PROCESSING_DATE,
    },
    'atletas_por_sexo': {
        'dataset_name': 'atletas_por_sexo',
        'layer': 'prata',
        'source_datasets': ['bronze/olympics_historico', 'bronze/olympics_paris2024'],
        'join_logic': 'UNION de agregacoes por genero/ano. Genero inferido de nomes de eventos no historico.',
        'derived_fields': ['gender (inferido)', 'has_medal', 'unique_athletes', 'total_participations', 'total_medals'],
        'data_quality_notes': 'Genero do historico e inferido e pode conter Mixed/Unknown.',
        'schema': get_schema_info(df_genero),
        'row_count': len(df_genero),
        'processing_date': PROCESSING_DATE,
    },
}

for name, meta in prata_metadata.items():
    save_metadata(meta, PRATA_DIR / f'{name}_metadata.json')

print('\nCamada PRATA concluida.')

---
## 4. Camada GOLD — Analytics

1. **Análise de Medalhas**: ranking global + visualização
2. **Análise de Modalidades**: evolução dos esportes
3. **Análise de Gênero**: participação por sexo

### 4.1 Análise de Medalhas — medalhas_summary

In [ ]:
medalhas_summary = df_medalhas.groupby('country_noc').agg(
    gold=('gold', 'sum'),
    silver=('silver', 'sum'),
    bronze=('bronze', 'sum'),
    total=('total', 'sum'),
    editions_participated=('edition', 'nunique'),
).reset_index()

# Adicionar nome do pais
country_map = df_tally_hist.drop_duplicates('country_noc')[['country_noc', 'country']].dropna()
medalhas_summary = medalhas_summary.merge(country_map, on='country_noc', how='left')

medalhas_summary = medalhas_summary.sort_values(
    ['gold', 'silver', 'bronze', 'total'], ascending=False
).reset_index(drop=True)
medalhas_summary['ranking'] = medalhas_summary.index + 1

medalhas_summary = medalhas_summary[[
    'ranking', 'country_noc', 'country', 'gold', 'silver', 'bronze', 'total', 'editions_participated'
]]

print(f'medalhas_summary: {len(medalhas_summary)} paises')

gold_medalhas_dir = GOLD_DIR / 'analise_medalhas'
medalhas_summary.to_csv(gold_medalhas_dir / 'medalhas_summary.csv', index=False)
print(f'Salvo: gold/analise_medalhas/medalhas_summary.csv')
medalhas_summary.head(15)

In [ ]:
# Visualizacao: Top 15 paises
top15 = medalhas_summary.head(15).copy()

fig, ax = plt.subplots(figsize=(14, 8))
x = range(len(top15))
width = 0.25

ax.bar([i - width for i in x], top15['gold'], width, label='Ouro', color='#FFD700')
ax.bar(x, top15['silver'], width, label='Prata', color='#C0C0C0')
ax.bar([i + width for i in x], top15['bronze'], width, label='Bronze', color='#CD7F32')

ax.set_xlabel('Pais', fontsize=12)
ax.set_ylabel('Numero de Medalhas', fontsize=12)
ax.set_title('Top 15 Paises - Medalhas Olimpicas (1896-2024)', fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(top15['country_noc'], rotation=45, ha='right')
ax.legend()

plt.tight_layout()
plot_path = gold_medalhas_dir / 'medalhas_plot.png'
fig.savefig(plot_path, dpi=150, bbox_inches='tight')
print(f'Grafico salvo: gold/analise_medalhas/medalhas_plot.png')
plt.show()

In [ ]:
# Metadados medalhas
medalhas_gold_metadata = {
    'dataset_name': 'medalhas_summary',
    'layer': 'gold',
    'source_datasets': ['prata/medalhas_1986_2024'],
    'kpi_definitions': {
        'ranking': 'Posicao no ranking global (ouro > prata > bronze > total).',
        'gold': 'Total acumulado de medalhas de ouro (1896-2024).',
        'silver': 'Total acumulado de medalhas de prata (1896-2024).',
        'bronze': 'Total acumulado de medalhas de bronze (1896-2024).',
        'total': 'Soma de todas as medalhas.',
        'editions_participated': 'Edicoes olimpicas com pelo menos 1 medalha.',
    },
    'aggregation_logic': 'SUM gold/silver/bronze/total por country_noc. COUNT DISTINCT edition. Ranking por gold DESC.',
    'assumptions': [
        'Dados de 2024 baseados em medalhas individuais agregadas.',
        'Codigos NOC podem mudar ao longo do tempo (USSR/RUS/ROC/AIN).',
        'Ranking segue convencao olimpica: prioridade para ouro.',
    ],
    'outputs': ['medalhas_summary.csv', 'medalhas_plot.png'],
    'schema': get_schema_info(medalhas_summary),
    'row_count': len(medalhas_summary),
    'processing_date': PROCESSING_DATE,
}
save_metadata(medalhas_gold_metadata, gold_medalhas_dir / 'metadata.json')
print('Metadata analise medalhas salvo.')

### 4.2 Análise de Modalidades

In [ ]:
gold_mod_dir = GOLD_DIR / 'analise_modalidades'
ensure_dir(gold_mod_dir)

modalidades_summary = df_modalidades.groupby(['year', 'edition']).agg(
    total_sports=('sport', 'nunique'),
    total_events=('event', 'nunique'),
).reset_index().sort_values('year')

modalidades_summary.to_csv(gold_mod_dir / 'modalidades_summary.csv', index=False)
print(f'Salvo: gold/analise_modalidades/modalidades_summary.csv')

# Visualizacao
fig, ax1 = plt.subplots(figsize=(14, 6))
summer = modalidades_summary[modalidades_summary['edition'].str.contains('Summer')]
winter = modalidades_summary[modalidades_summary['edition'].str.contains('Winter')]

ax1.plot(summer['year'], summer['total_sports'], 'o-', color='#E63946', label='Esportes (Verao)', linewidth=2)
ax1.plot(winter['year'], winter['total_sports'], 's--', color='#457B9D', label='Esportes (Inverno)', linewidth=2)

ax1.set_xlabel('Ano', fontsize=12)
ax1.set_ylabel('Numero de Esportes', fontsize=12)
ax1.set_title('Evolucao do Numero de Esportes Olimpicos (1896-2024)', fontsize=14, fontweight='bold')
ax1.legend(loc='upper left')

plt.tight_layout()
fig.savefig(gold_mod_dir / 'modalidades_plot.png', dpi=150, bbox_inches='tight')
print(f'Grafico salvo: gold/analise_modalidades/modalidades_plot.png')
plt.show()
modalidades_summary.tail(10)

In [ ]:
modalidades_gold_metadata = {
    'dataset_name': 'modalidades_summary',
    'layer': 'gold',
    'source_datasets': ['prata/modalidades_1986_2024'],
    'kpi_definitions': {
        'total_sports': 'Numero de esportes distintos por edicao.',
        'total_events': 'Numero de eventos distintos por edicao.',
    },
    'aggregation_logic': 'COUNT DISTINCT sport e event por year/edition.',
    'assumptions': [
        'Nomes de esportes podem variar entre fontes.',
        'Dados de 2024 refletem apenas eventos com medalhistas.',
    ],
    'outputs': ['modalidades_summary.csv', 'modalidades_plot.png'],
    'schema': get_schema_info(modalidades_summary),
    'row_count': len(modalidades_summary),
    'processing_date': PROCESSING_DATE,
}
save_metadata(modalidades_gold_metadata, gold_mod_dir / 'metadata.json')
print('Metadata analise modalidades salvo.')

### 4.3 Análise de Gênero

In [ ]:
gold_gen_dir = GOLD_DIR / 'analise_genero'
ensure_dir(gold_gen_dir)

df_gen_viz = df_genero[df_genero['gender'].isin(['Male', 'Female'])].copy()
df_gen_viz_summer = df_gen_viz[df_gen_viz['edition'].str.contains('Summer', na=False)]

genero_summary = df_gen_viz_summer.groupby(['year', 'gender']).agg(
    total_participations=('total_participations', 'sum'),
    unique_athletes=('unique_athletes', 'sum'),
    total_medals=('total_medals', 'sum'),
).reset_index()

genero_summary.to_csv(gold_gen_dir / 'genero_summary.csv', index=False)
print(f'Salvo: gold/analise_genero/genero_summary.csv')

# Visualizacao
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

for gender, color in [('Male', '#1D3557'), ('Female', '#E63946')]:
    mask = genero_summary['gender'] == gender
    data = genero_summary[mask].sort_values('year')
    ax1.plot(data['year'], data['unique_athletes'], 'o-', color=color, label=gender, linewidth=2, markersize=3)

ax1.set_xlabel('Ano', fontsize=11)
ax1.set_ylabel('Atletas Unicos', fontsize=11)
ax1.set_title('Atletas por Genero - Jogos de Verao', fontsize=13, fontweight='bold')
ax1.legend()

pivot = genero_summary.pivot_table(index='year', columns='gender', values='unique_athletes', aggfunc='sum').fillna(0)
if 'Female' in pivot.columns and 'Male' in pivot.columns:
    pivot['pct_female'] = pivot['Female'] / (pivot['Female'] + pivot['Male']) * 100
    ax2.fill_between(pivot.index, pivot['pct_female'], alpha=0.3, color='#E63946')
    ax2.plot(pivot.index, pivot['pct_female'], 'o-', color='#E63946', linewidth=2, markersize=3)
    ax2.axhline(y=50, color='gray', linestyle='--', alpha=0.5)
    ax2.set_ylabel('% Feminino', fontsize=11)

ax2.set_xlabel('Ano', fontsize=11)
ax2.set_title('Percentual Feminino - Jogos de Verao', fontsize=13, fontweight='bold')

plt.tight_layout()
fig.savefig(gold_gen_dir / 'genero_plot.png', dpi=150, bbox_inches='tight')
print(f'Grafico salvo: gold/analise_genero/genero_plot.png')
plt.show()
genero_summary.tail(10)

In [ ]:
genero_gold_metadata = {
    'dataset_name': 'genero_summary',
    'layer': 'gold',
    'source_datasets': ['prata/atletas_por_sexo'],
    'kpi_definitions': {
        'total_participations': 'Total de participacoes em eventos.',
        'unique_athletes': 'Numero de atletas distintos.',
        'total_medals': 'Total de medalhas conquistadas.',
    },
    'aggregation_logic': 'SUM participacoes/atletas/medalhas por year/gender. Filtrado Jogos de Verao.',
    'assumptions': [
        'Genero do historico inferido de nomes de eventos (Men/Women/Ladies).',
        'Eventos mistos excluidos da visualizacao.',
        'Dados de 2024 usam campo gender diretamente.',
    ],
    'outputs': ['genero_summary.csv', 'genero_plot.png'],
    'schema': get_schema_info(genero_summary),
    'row_count': len(genero_summary),
    'processing_date': PROCESSING_DATE,
}
save_metadata(genero_gold_metadata, gold_gen_dir / 'metadata.json')
print('Metadata analise genero salvo.')

---
## 5. metadata_schema.json — Schema Padrão

In [ ]:
metadata_schema = {
    '$schema': 'http://json-schema.org/draft-07/schema#',
    'title': 'Olympic Data Lake Metadata Schema',
    'description': 'Schema padrao para todos os arquivos de metadados do Data Lake Olimpico.',
    'type': 'object',
    'required': ['dataset_name', 'layer', 'processing_date'],
    'properties': {
        'dataset_name': {'type': 'string', 'description': 'Nome identificador do dataset.'},
        'layer': {'type': 'string', 'enum': ['raw', 'bronze', 'prata', 'gold'], 'description': 'Camada do Data Lake.'},
        'schema': {
            'type': 'array',
            'description': 'Lista de colunas com tipos e contagem de nao-nulos.',
            'items': {
                'type': 'object',
                'properties': {
                    'column': {'type': 'string'},
                    'dtype': {'type': 'string'},
                    'non_null_count': {'type': 'integer'},
                },
            },
        },
        'source': {'type': 'string', 'description': 'Fonte original dos dados.'},
        'source_datasets': {'type': 'array', 'items': {'type': 'string'}, 'description': 'Datasets de origem.'},
        'description': {'type': 'string', 'description': 'Descricao do dataset.'},
        'main_fields': {'type': 'array', 'items': {'type': 'string'}, 'description': 'Campos principais.'},
        'transformations_applied': {'type': 'array', 'items': {'type': 'string'}, 'description': 'Transformacoes aplicadas.'},
        'join_logic': {'type': 'string', 'description': 'Logica de JOIN (prata/gold).'},
        'derived_fields': {'type': 'array', 'items': {'type': 'string'}, 'description': 'Campos derivados.'},
        'data_quality_notes': {'type': 'string', 'description': 'Notas sobre qualidade.'},
        'quality_checks': {'type': 'array', 'items': {'type': 'string'}, 'description': 'Validacoes de qualidade.'},
        'lineage': {
            'type': 'object',
            'description': 'Informacoes de linhagem.',
            'properties': {
                'upstream': {'type': 'array', 'items': {'type': 'string'}},
                'downstream': {'type': 'array', 'items': {'type': 'string'}},
            },
        },
        'kpi_definitions': {'type': 'object', 'description': 'Definicoes de KPIs (gold).'},
        'aggregation_logic': {'type': 'string', 'description': 'Logica de agregacao.'},
        'assumptions': {'type': 'array', 'items': {'type': 'string'}, 'description': 'Premissas.'},
        'row_count': {'type': 'integer', 'description': 'Numero de linhas.'},
        'column_count': {'type': 'integer', 'description': 'Numero de colunas.'},
        'collection_date': {'type': 'string', 'description': 'Data de coleta.'},
        'processing_date': {'type': 'string', 'description': 'Data/hora de processamento.'},
        'observations': {'type': 'string', 'description': 'Observacoes adicionais.'},
        'outputs': {'type': 'array', 'items': {'type': 'string'}, 'description': 'Arquivos de saida.'},
    },
}

schema_path = DATALAKE_DIR / 'metadata_schema.json'
save_metadata(metadata_schema, schema_path)
print(f'metadata_schema.json salvo.')

---
## 6. README.md — Documentação

In [ ]:
readme_content = r'''# Olympic Data Lake - Pipeline de Dados Olimpicos

## Visao Geral

Este projeto implementa um **Data Lake estruturado** para dados olimpicos,
cobrindo desde os primeiros Jogos Olimpicos modernos (1896) ate as Olimpiadas de Paris 2024.

O pipeline segue a arquitetura **RAW > BRONZE > PRATA (SILVER) > GOLD**.

---

## Arquitetura - Camadas do Data Lake

| Camada | Descricao | Formato |
|--------|-----------|--------|
| **RAW** | Dados brutos originais | CSV, JSON |
| **BRONZE** | Dados padronizados (snake_case, trimming, parquet) | Parquet, JSON |
| **PRATA** | Dados integrados (JOINs, deduplicacao) | Parquet, JSON |
| **GOLD** | Dados analiticos (agregacoes, KPIs, graficos) | CSV, PNG, JSON |

---

## Estrutura de Pastas

```
olympics-datalake/
|
|-- README.md
|-- metadata_schema.json
|
|-- raw/
|   |-- olympics_historico.csv
|   |-- olympics_paris2024.csv
|   |-- olympics_historico.json
|   |-- olympics_paris2024.json
|   |-- medal_tally_historico.csv
|   |-- result_historico.csv
|   |-- medals_paris2024.csv
|   +-- *_metadata.json
|
|-- bronze/
|   |-- *.parquet
|   +-- *_metadata.json
|
|-- prata/
|   |-- medalhas_1986_2024.parquet
|   |-- modalidades_1986_2024.parquet
|   |-- atletas_por_sexo.parquet
|   +-- *_metadata.json
|
+-- gold/
    |-- analise_medalhas/
    |   |-- medalhas_summary.csv
    |   |-- medalhas_plot.png
    |   +-- metadata.json
    |-- analise_modalidades/
    |   |-- modalidades_summary.csv
    |   |-- modalidades_plot.png
    |   +-- metadata.json
    +-- analise_genero/
        |-- genero_summary.csv
        |-- genero_plot.png
        +-- metadata.json
```

---

## Tecnologias Utilizadas

- **Python 3.8+**
- **pandas** - Manipulacao e analise de dados
- **pyarrow** - Leitura/escrita de Parquet
- **matplotlib / seaborn** - Visualizacoes
- **pathlib / os** - Gerenciamento de caminhos
- **json** - Metadados estruturados
- **Jupyter Notebook** - Documentacao executavel

---

## Como Executar

1. **Pre-requisitos**:
   ```bash
   pip install pandas pyarrow matplotlib seaborn
   ```

2. **Estrutura esperada de dados de entrada**:
   ```
   raw/
   |-- 1986-2022/
   |   |-- world_olympedia_olympics_athlete_event_result.csv
   |   |-- world_olympedia_olympics_game_medal_tally(4).csv
   |   +-- world_olympedia_olympics_result(2).csv
   +-- 2024/
       |-- medallists.csv
       +-- medals(1).csv
   ```

3. **Executar o notebook**:
   ```bash
   jupyter notebook datalake_pipeline.ipynb
   ```
   Execute todas as celulas sequencialmente. O pipeline e **idempotente**.

---

## Fontes de Dados

| Dataset | Fonte | Periodo | Descricao |
|---------|-------|---------|----------|
| athlete_event_result | Olympedia | 1896-2022 | Resultados individuais |
| game_medal_tally | Olympedia | 1896-2022 | Quadro de medalhas por pais/edicao |
| result | Olympedia | 1896-2022 | Detalhes de eventos |
| medallists | Kaggle | 2024 | Medalhistas de Paris 2024 |
| medals | Kaggle | 2024 | Medalhas de Paris 2024 |

---

## Exemplos de Saidas

### Top 15 Paises por Medalhas (1896-2024)
![Medalhas Plot](gold/analise_medalhas/medalhas_plot.png)

### Evolucao dos Esportes Olimpicos
![Modalidades Plot](gold/analise_modalidades/modalidades_plot.png)

### Participacao por Genero
![Genero Plot](gold/analise_genero/genero_plot.png)

---

## Metadados

Cada dataset possui um arquivo `*_metadata.json` com:
- Schema (colunas e tipos)
- Fonte dos dados
- Transformacoes aplicadas
- Contagem de linhas
- Data de processamento
- Logica de agregacao/JOIN (quando aplicavel)

Schema padrao em `metadata_schema.json`.
'''

readme_path = DATALAKE_DIR / 'README.md'
with open(readme_path, 'w', encoding='utf-8') as f:
    f.write(readme_content)

print(f'README.md salvo.')

---
## 7. Copiar Notebook para Gold

In [ ]:
notebook_src = BASE_DIR / 'datalake_pipeline.ipynb'
notebook_dst = GOLD_DIR / 'analise_medalhas' / 'notebook.ipynb'

if notebook_src.exists():
    shutil.copy2(notebook_src, notebook_dst)
    print(f'Notebook copiado para: gold/analise_medalhas/notebook.ipynb')
else:
    print(f'Notebook fonte nao encontrado em {notebook_src}. Copie manualmente apos salvar.')

---
## 8. Validação Final

In [ ]:
print('=' * 60)
print('VALIDACAO FINAL - Estrutura do Data Lake')
print('=' * 60)

expected_structure = {
    'README.md': DATALAKE_DIR / 'README.md',
    'metadata_schema.json': DATALAKE_DIR / 'metadata_schema.json',
    'raw/olympics_historico.csv': RAW_DIR / 'olympics_historico.csv',
    'raw/olympics_paris2024.csv': RAW_DIR / 'olympics_paris2024.csv',
    'raw/olympics_historico.json': RAW_DIR / 'olympics_historico.json',
    'raw/olympics_paris2024.json': RAW_DIR / 'olympics_paris2024.json',
    'bronze/olympics_historico.parquet': BRONZE_DIR / 'olympics_historico.parquet',
    'bronze/olympics_paris2024.parquet': BRONZE_DIR / 'olympics_paris2024.parquet',
    'bronze/medal_tally_historico.parquet': BRONZE_DIR / 'medal_tally_historico.parquet',
    'prata/medalhas_1986_2024.parquet': PRATA_DIR / 'medalhas_1986_2024.parquet',
    'prata/modalidades_1986_2024.parquet': PRATA_DIR / 'modalidades_1986_2024.parquet',
    'prata/atletas_por_sexo.parquet': PRATA_DIR / 'atletas_por_sexo.parquet',
    'gold/analise_medalhas/medalhas_summary.csv': GOLD_DIR / 'analise_medalhas' / 'medalhas_summary.csv',
    'gold/analise_medalhas/medalhas_plot.png': GOLD_DIR / 'analise_medalhas' / 'medalhas_plot.png',
    'gold/analise_medalhas/metadata.json': GOLD_DIR / 'analise_medalhas' / 'metadata.json',
    'gold/analise_modalidades/modalidades_summary.csv': GOLD_DIR / 'analise_modalidades' / 'modalidades_summary.csv',
    'gold/analise_modalidades/metadata.json': GOLD_DIR / 'analise_modalidades' / 'metadata.json',
    'gold/analise_genero/genero_summary.csv': GOLD_DIR / 'analise_genero' / 'genero_summary.csv',
    'gold/analise_genero/metadata.json': GOLD_DIR / 'analise_genero' / 'metadata.json',
}

all_ok = True
for label, path in expected_structure.items():
    exists = path.exists()
    status = 'OK' if exists else 'MISSING'
    if not exists:
        all_ok = False
    print(f'  [{status}] {label}')

print(f'\n{"="*60}')
if all_ok:
    print('TODOS OS ARTEFATOS CRIADOS COM SUCESSO!')
else:
    print('ALGUNS ARTEFATOS NAO FORAM ENCONTRADOS.')
print(f'{"="*60}')

In [ ]:
# Resumo de armazenamento
print('\nResumo de Armazenamento por Camada:')
print('-' * 45)
for layer_name, layer_dir in [('raw', RAW_DIR), ('bronze', BRONZE_DIR), ('prata', PRATA_DIR), ('gold', GOLD_DIR)]:
    total_size = sum(f.stat().st_size for f in layer_dir.rglob('*') if f.is_file())
    file_count = sum(1 for f in layer_dir.rglob('*') if f.is_file())
    print(f'  {layer_name:8s}: {file_count:3d} arquivos | {total_size / 1024 / 1024:.2f} MB')

total = sum(f.stat().st_size for f in DATALAKE_DIR.rglob('*') if f.is_file())
print(f'  {"TOTAL":8s}: {sum(1 for f in DATALAKE_DIR.rglob("*") if f.is_file()):3d} arquivos | {total / 1024 / 1024:.2f} MB')
print('\nPipeline concluido com sucesso!')